# Tâche 2 - Classification automatique de descriptions d'incidents

Cette tâche consiste à classifier de courts textes décrivant des incidents sur des chantiers de construction. Pour chaque incident, on retrouve une étiquette qui décrit un aspect de l’incident. Nous avons substitué les étiquettes de classes par des nombres entiers. Ces étiquettes sont inconnues et vous devrez tenter de les identifier à la section 3 de ce *notebook*. 

Pour la première partie, vous devez compléter la fonction (*train_and_test_classifier*) qui entraîne un modèle (les options étant la régression logistique et le naïf bayésien) et en faire l'évaluation sur des données d'entraînement et des données de test. Pour la deuxième partie, vous devez tentez de déterminer à quoi correspond chacune des classes d’incident. 

## Section 1 - Lecture des fichiers de données

La fonction *load_incident_data* est utilisée pour lire les 2 fichiers (*train* et *test*). Rien à modifier dans cette section. 

In [1]:
import json

train_json_fn = "./data/incidents_train.json"
test_json_fn = "./data/incidents_test.json"


def load_incident_dataset(filename):
    with open(filename, 'r') as fp:
        incident_list = json.load(fp)
    return incident_list

In [2]:
train_list = load_incident_dataset(train_json_fn)
print("Nombre d'incidents:", len(train_list))
print("\nUn exemple:\n", train_list[10])

Nombre d'incidents: 2475

Un exemple:
 {'text': " At approximately 12:15 p.m. on February 8  2011  Employee # 1 and four other  coworkers were working in the West storage Facility at Enterprise Products  located at Mont Belvieu  Texas. A major fire began but Employee # 1 wasn't  able to escape. The recovery effort was delayed until the evening of February  10  2011  due to the presence of hydrocarbon vapors and small residual fires.   ", 'label': '6'}


In [9]:
test_list = load_incident_dataset(test_json_fn)
print("Nombre d'incidents", len(test_list))
incident = test_list[10]
print("\nUne description d'incident:", incident["text"])
print("\nSon étiquette:", incident["label"])

Nombre d'incidents 531

Une description d'incident:  On May 24  2004  Employee #1 was welding the deck caps on the of fourth floor  decking. The employee fell from the edge and sustained life threatening head  and internal injuries. Employee #1 was then transferred to the Kaweah Delta  District Hospital  where he subsequently died from his injuries. The employee  was not wearing any fall protection but there was a single line of wire rope  at the perimeter of the structure.                                              

Son étiquette: 5


## Section 2 - Entraînement et évaluation des modèles  

Vous pouvez ajouter tout le code dont vous avez besoin pour l'entraînement. Merci de ne pas modifier la signature de la fonction d'entraînement. Vous pouvez ajouter des cellules au *notebook* si nécessaire. 

In [23]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

In [24]:
def train_and_test_classifier(train_fn, test_fn, model='NB'):
    """
    :param train_fn et test_fn: les 2 fichiers utilisées pour entraîner et tester les classificateurs.
    :param model: le type de classificateur. NB = Naive Bayes, LR = Régression logistique.
    :return: un dictionnaire contenant 3 valeurs: l'accuracy à l'entraînement, l'accuracy sur le jeu de test, 
             et la matrice de confusion calculée sur les données de test.
             Vous pouvez ajouter d'autres éléments à results, par ex. les modèles entraînés.
    """

    # Charger les données d'entraînement et de test
    train_list = load_incident_dataset(train_fn)
    test_list = load_incident_dataset(test_fn)

    # Préparer les données pour l'entraînement et le test
    train_texts = [incident['text'] for incident in train_list]
    train_labels = [incident['label'] for incident in train_list]
    test_texts = [incident['text'] for incident in test_list]
    test_labels = [incident['label'] for incident in test_list]

    # Vectoriser les textes
    vectorizer = CountVectorizer()
    X_train = vectorizer.fit_transform(train_texts)
    X_test = vectorizer.transform(test_texts)

    # Initialiser le classificateur
    if model == 'NB':
        classifier = MultinomialNB()
    elif model == 'LR':
        classifier = LogisticRegression(max_iter=1000)

    # Entraîner le classificateur
    classifier.fit(X_train, train_labels)

    # Faire des prédictions
    y_train_pred = classifier.predict(X_train)
    y_test_pred = classifier.predict(X_test)

    # Calculer les métriques
    accuracy_train = accuracy_score(train_labels, y_train_pred)
    accuracy_test = accuracy_score(test_labels, y_test_pred)
    confusion_mat = confusion_matrix(test_labels, y_test_pred)

    # Les résultats à retourner
    results = dict()
    results['accuracy_train'] = accuracy_train
    results['accuracy_test'] = accuracy_test
    results['confusion_matrix'] = confusion_mat

    results['model'] = classifier
    results['vectorizer'] = vectorizer

    return results


print("\nEntraînement et test du classificateur Naive Bayes...")
results_nb = train_and_test_classifier(train_json_fn, test_json_fn, model='NB')
print("Accuracy à l'entraînement:", results_nb['accuracy_train'])
print("Accuracy sur le jeu de test:", results_nb['accuracy_test']) 


Entraînement et test du classificateur Naive Bayes...
Accuracy à l'entraînement: 0.8569696969696969
Accuracy sur le jeu de test: 0.6760828625235404


In [25]:
print("\nEntraînement et test du classificateur Naive Bayes...")
results_nb = train_and_test_classifier(train_json_fn, test_json_fn, model='NB')
print("Accuracy à l'entraînement:", results_nb['accuracy_train'])
print("Accuracy sur le jeu de test:", results_nb['accuracy_test']) 

print("\nEntraînement et test de la Régression Logistique...")
results_lr = train_and_test_classifier(train_json_fn, test_json_fn, model='LR')
print("Accuracy à l'entraînement:", results_lr['accuracy_train'])
print("Accuracy sur le jeu de test:", results_lr['accuracy_test'])


Entraînement et test du classificateur Naive Bayes...
Accuracy à l'entraînement: 0.8569696969696969
Accuracy sur le jeu de test: 0.6760828625235404

Entraînement et test de la Régression Logistique...
Accuracy à l'entraînement: 1.0
Accuracy sur le jeu de test: 0.6930320150659134


## Section 3 - À quoi correspondent les classes? Explicabilité du modèle

Vous pouvez ajouter tout le code et toutes les cellules dont vous avez besoin pour cette sous-tâche.

In [31]:
import numpy as np

modele_lr = results_lr['model']
vectorizer = results_lr['vectorizer']

mots_vocabulaire = vectorizer.get_feature_names_out()


print("Mots les plus importants par classe (Régression Logistique) :\n")

for i, classe in enumerate(modele_lr.classes_):
    poids = modele_lr.coef_[i]
    
    indices_top_mots = np.argsort(poids)[::-1][:10] 
    
    top_mots = [mots_vocabulaire[idx] for idx in indices_top_mots]
    
    print(f"Classe {classe} : {', '.join(top_mots)}")


Mots les plus importants par classe (Régression Logistique) :

Classe 0 : collapsed, crushed, vehicle, trench, garage, run, crushing, ran, wall, loader
Classe 1 : drowned, barge, into, river, diver, coffer, dam, pool, boat, 12
Classe 2 : electrocuted, energized, contacted, power, volt, shock, contact, overhead, bus, electrical
Classe 3 : asphyxia, gas, asphyxiated, overcome, methylene, chloride, paint, 2011, inhalation, carbon
Classe 4 : heat, burned, hot, temperature, degrees, worker, fahrenheit, 2005, insulation, steam
Classe 5 : fell, balance, ladder, ft, fall, feet, scaffold, height, opening, fractured
Classe 6 : burns, fire, ignited, gas, explosion, exploded, tank, natural, fuel, hospitalized
Classe 7 : saw, finger, case, about, burns, hopper, serial, struck, directing, tools
Classe 8 : struck, striking, elevator, blocking, sustained, eye, beam, march, counterweight, load


In [30]:
modele_nb = results_nb['model']
vectorizer_nb = results_nb['vectorizer']

mots_vocabulaire_nb = vectorizer_nb.get_feature_names_out()

print("\nMots les plus importants par classe (Naive Bayes) :\n")

for i, classe in enumerate(modele_nb.classes_):
    # C'est ici que ça change pour le Naïf Bayésien
    poids = modele_nb.feature_log_prob_[i]
    
    indices_top_mots = np.argsort(poids)[::-1][:10] 
    
    top_mots = [mots_vocabulaire_nb[idx] for idx in indices_top_mots]
    
    print(f"Classe {classe} : {', '.join(top_mots)}")



Mots les plus importants par classe (Naive Bayes) :

Classe 0 : the, was, and, employee, to, of, on, in, he, at
Classe 1 : the, employee, was, and, to, on, of, in, he, water
Classe 2 : the, was, and, employee, to, of, on, he, an, in
Classe 3 : the, and, to, employee, was, of, on, in, he, were
Classe 4 : the, and, was, to, employee, he, of, at, on, his
Classe 5 : the, was, employee, and, to, of, on, he, at, in
Classe 6 : the, and, was, employee, to, of, on, he, in, an
Classe 7 : the, was, employee, and, to, of, on, he, in, his
Classe 8 : the, was, and, to, employee, of, on, in, he, at


## Section 4 - Section réservée pour nos tests